# Vitrine 01 (Nível Júnior): Geomarketing e Otimização de Rede
**Foco:** Análise Descritiva, Infraestrutura de Postos e Oportunidades de Expansão.

Este notebook demonstra a capacidade de extrair valor estratégico de dados operacionais e geográficos, identificando gaps de mercado e alvos prioritários para investimento (CAPEX).

In [ ]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import numpy as np

import warnings
warnings.filterwarnings('ignore')

# Configurações de exibição
pd.set_option('display.max_columns', None)

DATA_PATH = '../data/processed/dados_limpos.csv'
df = pd.read_csv(DATA_PATH)
df.head()

## 1. Concentração de Mercado (Análise de Marcas)
Identificar os grandes players e a fragmentação do mercado é o primeiro passo para entender o cenário competitivo.

In [ ]:
brand_counts = df['brand_name'].value_counts().reset_index()
brand_counts.columns = ['brand_name', 'count']
brand_counts['cumulative_perc'] = brand_counts['count'].cumsum() / brand_counts['count'].sum() * 100

top_15 = brand_counts.head(15)

fig_pareto = px.bar(top_15, x='brand_name', y='count',
                     title='<b>Predomínio de Marcas:</b> Shell e Esso Lideram o Market Share de Pontos',
                     labels={'brand_name': 'Marca', 'count': 'Total de Postos'},
                     color='count', color_continuous_scale='Greens')

fig_pareto.update_layout(template='plotly_white')
fig_pareto.show()

## 2. Diagnóstico de Infraestrutura e Gaps de Conveniência
Onde estão os postos que ainda não oferecem serviços modernos como Carregamento EV ou banheiros? Estes são alvos para 'Retrofit' (modernização).

In [ ]:
top_counties = df['county'].value_counts().head(8).index
df_top = df[df['county'].isin(top_counties)]

infra_stats = df_top.groupby('county')[['has_ev_charging', 'has_customer_toilets', 'has_car_wash']].mean() * 100
infra_stats = infra_stats.reset_index().melt(id_vars='county', var_name='Serviço', value_name='Disponibilidade_Perc')

fig_infra = px.bar(infra_stats, x='county', y='Disponibilidade_Perc', color='Serviço', barmode='group',
                    title='<b>Déficit Regional:</b> A Baixa Adoção de Carregadores EV Representa um Gap de Mercado',
                    labels={'county': 'Região', 'Disponibilidade_Perc': 'Disponibilidade (%)'},
                    color_discrete_map={'has_ev_charging': '#2ecc71', 'has_customer_toilets': '#3498db', 'has_car_wash': '#95a5a6'})

fig_infra.update_layout(template='plotly_white', yaxis_range=[0,100])
fig_infra.show()

## 3. Inteligência para M&A: Identificação de Ativos Inativos
Postos temporariamente fechados são oportunidades de aquisição com menor custo de entrada para grandes redes.

In [ ]:
closed_stations = df[df['is_temporarily_closed'] | df['is_permanently_closed']]
ma_summary = closed_stations.groupby('city').size().reset_index(name='count').sort_values('count', ascending=False).head(10)

print(f'Total de Alvos M&A Identificados: {len(closed_stations)}')
ma_summary

### Conclusão e Próximos Passos
1. **Modernização Digital:** Focar em regiões com alto tráfego mas baixo EV Charging.
2. **Aquisição:** Priorizar auditoria nos postos inativos das cidades identificadas.
3. **Expansão:** Cidades com baixa densidade de postos 24h são oportunidades de 'Oceano Azul'.